In [1]:
import matplotlib.pyplot as plt
import numpy as np
import itk
import napari
import topometrics.leaderboard
from topometrics.toposcore import TopoScore
import scipy.ndimage as ndimage
from skimage.morphology import remove_small_objects, skeletonize
from ipywidgets import interact, IntSlider
import cc3d
# import cv2

In [2]:
CASE_STUDY = "956073442"


In [ ]:

image = itk.imread(f"{CASE_STUDY}_image.tif", itk.F)
label = itk.imread(f"{CASE_STUDY}_label.tif", itk.F)
pred  = itk.imread(f"{CASE_STUDY}_pred.tif", itk.F)


# Check volume size
size = itk.size(image)
print("Volume size:", size)

In [ ]:
imagearr = itk.GetArrayFromImage(image)
labelarr = itk.GetArrayFromImage(label)
predarr  = itk.GetArrayFromImage(pred)

In [ ]:
surface_tolerance: float = 2.0
voi_connectivity: int = 26
voi_transform: str = 'one_over_one_plus'
voi_alpha: float = 0.3
topo_weight: float = 0.3
surface_dice_weight: float = 0.35
voi_weight: float = 0.35


def calc_report(y, y_hat):
    score_report = topometrics.leaderboard.compute_leaderboard_score(
        predictions=y_hat,
        labels=y,
        dims=(0, 1, 2),
        spacing=(1.0, 1.0, 1.0),  # (z, y, x)
        surface_tolerance=surface_tolerance,  # in spacing units
        voi_connectivity=voi_connectivity,
        voi_transform=voi_transform,
        voi_alpha=voi_alpha,
        combine_weights=(topo_weight, surface_dice_weight, voi_weight),  # (Topo, SurfaceDice, VOI)
        fg_threshold=None,  # None => legacy "!= 0"; else uses "x > threshold"
        ignore_label=2,  # voxels with this GT label are ignored
        ignore_mask=None,  # or pass an explicit boolean mask
    ) 
    return score_report

In [ ]:
calc_report(labelarr, predarr)

In [ ]:
ignore = labelarr == 2
labelarr[ignore] = 0
predarr[ignore] = 0

In [ ]:

def view_slice(SLICE):
    img3 = np.hstack([
        imagearr[SLICE],
        imagearr[:, SLICE],
        imagearr[:, :, SLICE]
    ])
    sk3 = np.hstack([
        skeletonize(predarr[SLICE]),
        skeletonize(predarr[:, SLICE]),
        skeletonize(predarr[:, :, SLICE])
    ])

    p3 = np.hstack([
        predarr[SLICE],
        predarr[:, SLICE],
        predarr[:, :, SLICE]
    ]) * 255

    l3 = np.hstack([
        labelarr[SLICE],
        labelarr[:, SLICE],
        labelarr[:, :, SLICE]
    ]) * 255
    # Assume 'skeleton' is your binary image
    kernel = np.ones((3,3), np.uint8)  # 5x5 square thickness
    sk3 = cv2.dilate(sk3.astype(np.uint8), kernel, iterations=1) * 255

    plt.figure(figsize=(8,8))
    plt.imshow(np.vstack([img3, p3, l3]), cmap='gray')
    plt.axis('off')
    plt.show()

In [ ]:
# # Create slider widget
# max_slice = imagearr.shape[0] - 1
# interact(view_slice, SLICE=IntSlider(min=0, max=max_slice, step=1, value=20));

In [ ]:
predarr = cc3d.dust( predarr, threshold=16, connectivity=6, in_place=True )

In [ ]:
calc_report(labelarr, predarr)

In [ ]:
struct_6 = np.array([[[0,0,0],[0,1,0],[0,0,0]], 
                     [[0,1,0],[1,1,1],[0,1,0]], 
                     [[0,0,0],[0,1,0],[0,0,0]]])
predarr, predcc = ndimage.label(predarr, structure=struct_6)
labelarr, labelcc = ndimage.label(labelarr, structure=struct_6)

In [ ]:
labelcc, predcc

In [ ]:

# Visualization

# Launch viewer
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(imagearr, name='volume', rendering='attenuated_mip')
# viewer.add_image(labelarr, name='label', rendering='attenuated_mip')
# viewer.add_image(predarr, name='pred', rendering='attenuated_mip')


for i in range(1, min(labelcc + 1, 10)):

   gtlayer = labelarr.copy()
   predlayer = predarr.copy()

   layermask = labelarr == i
  
   
   gtlayer[~layermask] = 0
   predlayer[~layermask] = 0
    
   viewer.add_image(gtlayer, name=f'layer_{i}', rendering='attenuated_mip', colormap="PiYG") 
   viewer.add_image(predlayer, name=f'predlayer_{i}', rendering='attenuated_mip', colormap="PiYG") 


napari.run()